# Hosting CrewAI multi-agent crew with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing multi-agent crew, using Amazon Bedrock AgentCore Runtime. 

We will focus on a CrewAI with Amazon Bedrock model example. For Strands Agents with Amazon Bedrock model check [here](../01-strands-with-bedrock-model) and for a Strands Agents with an OpenAI model check [here](../03-strands-with-openai-model).


### Tutorial Details

| Information | Details |
|:--------------------|:-----------------------------------------------------------------------------|
| Tutorial type | Conversational |
| Agent type | Multi-agent crew |
| Agentic Framework | CrewAI |
| LLM model | Anthropic Claude 3.5 Haiku |
| Tutorial components | Hosting agent on AgentCore Runtime. Using CrewAI and Amazon Bedrock Model |
| Tutorial vertical | Cross-vertical |
| Example complexity | Easy |
| SDK used | Amazon BedrockAgentCore Python SDK and boto3 |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing multi-agent crew to AgentCore runtime. 

For demonstration purposes, we will use a CrewAI crew using Amazon Bedrock models

In our example we will use a research crew with two agents: a researcher and an analyst.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using CrewAI

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* uv package manager
* AWS credentials
* Docker running

Further, we need to install a few dependencies: 
* Amazon Bedrock AgentCore SDK
* CrewAI 
* Langchain community package
* Duckduckgo search

We have packaged all necessary dependencies in a pyproject.toml file so they can be installed conveniently. 

In [ ]:
!pip install -r requirements.txt

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
from botocore.exceptions import ClientError
import boto3
import sys
import os
import json
import time

# Get the current notebook's directory
current_dir = os.path.dirname(
    os.path.abspath("__file__" if "__file__" in globals() else ".")
)

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.join(utils_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)
iam_client = boto3.client("iam")
codebuild_client = boto3.client("codebuild")
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
s3 = boto3.client("s3", region_name=region)
sts = boto3.client("sts")

agent_name = "crewai_agentcore"
bucket_name = f"crewaiagentcore-{sts.get_caller_identity()["Account"]}"


## Creating your multi-agent crew and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

In this guide, we’ll walk through creating a research crew that will help us research and analyze a topic, then create a comprehensive report. This practical example demonstrates how AI agents can collaborate to accomplish complex tasks. The example is adapted from a [getting started guide](https://docs.crewai.com/en/guides/crews/first-crew) provided directly by CrewAI.

The local architecture looks as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>


### Defining agents, tasks, crew

We will first create the artifacts defining a local CrewAI agent, including: 
* agents.yaml, defining the two agents involved in our crew
* tasks.yaml, defining the tasks to be executed by the agents in our crew
* crew.py, defining our crew consisting of agents working on tasks as defined
* main.py, our local entrypoint kicking off the crew run

In [ ]:
%%writefile research_crew/agents.py

from textwrap import dedent
from crewai import Agent, LLM
from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import BaseTool
from functools import lru_cache

# Configure Bedrock LLM
llm = LLM(
    model="bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0",
)

# Define Search Tool
class SearchTool(BaseTool):
    name: str = "Search"
    description: str = "Useful for searching the web for information."
    search: DuckDuckGoSearchRun = DuckDuckGoSearchRun()
    call_count: int = 0
    max_calls: int = 3

    def _run(self, query: str) -> str:
        if self.call_count >= self.max_calls:
            return "SearchTool limit reached. Please reason using existing info."
        try:
            self.call_count += 1
            return self.search.invoke(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"
        
# Define Agents
researcher = Agent(
    role=dedent("""
        Senior Research Specialist for {topic}
    """),
    goal=dedent("""
        Find comprehensive and accurate information about {topic}
        with a focus on recent developments and key insights
    """),
    backstory=dedent("""
        You are an experienced research specialist with a talent for
        finding relevant information from various sources. You excel at
        organizing information in a clear and structured manner, making
        complex topics accessible to others.
    """),
    tools=[SearchTool()],
    allow_delegation=False,
    verbose=True,
    max_iter=3,
    max_rpm=100,
    llm=llm
)

analyst = Agent(
    role=dedent("""
        Data Analyst and Report Writer for {topic}
    """),
    goal=dedent("""
        Analyze research findings and create a comprehensive, well-structured
        report that presents insights in a clear and engaging way
    """),
    backstory=dedent("""
        You are a skilled analyst with a background in data interpretation
        and technical writing. You have a talent for identifying patterns
        and extracting meaningful insights from research data, then
        communicating those insights effectively through well-crafted reports.
    """),
    tools=[],
    allow_delegation=False,
    verbose=True,
    max_iter=3,
    max_rpm=100,
    llm=llm
)


In [ ]:
%%writefile research_crew/tasks.py

from crewai import Task
from textwrap import dedent
from .agents import researcher, analyst

# Define Tasks
task_1 = Task(
    description=dedent("""
        Conduct thorough research on {topic}. Focus on:
        1. Key concepts and definitions
        2. Historical development and recent trends
        3. Major challenges and opportunities
        4. Notable applications or case studies
        5. Future outlook and potential developments
    """),
    expected_output=dedent("""
        A comprehensive research document with well-organized sections covering
        all the requested aspects of {topic}. Include specific facts, figures,
        and examples where relevant.
    """),
    agent=researcher,
)

task_2 = Task(
    description=dedent("""
        Analyze the research findings and create a comprehensive report on {topic}.
        Your report should:
        1. State the topic and begin with an executive summary
        2. Include all key information from the research
        3. Provide insightful analysis of trends and patterns
        4. Offer recommendations or future considerations
        5. Be formatted in a professional, easy-to-read style with clear headings
    """),
    expected_output=dedent("""
        A polished, professional report on {topic} that presents the research
        findings with added analysis and insights. The report should be well-structured
        with an executive summary, main sections, and conclusion.
    """),
    agent=analyst,
    context=[task_1],
)

Lets create an S3 bucket to store final markdown report from crew.

In [ ]:
try:
    # Check if it already exists
    s3.head_bucket(Bucket=bucket_name)
    print(f"Bucket already exists: {bucket_name}")
except ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "404":
        # Create the bucket
        print(f"Creating bucket: {bucket_name}")
        if region == "us-east-1":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
    else:
        raise RuntimeError(f"Error checking/creating bucket: {e}")


In [ ]:
%%writefile local_crew.py

import datetime
import boto3
from botocore.exceptions import ClientError
import os

from crewai import Crew, Process
from research_crew.agents import researcher, analyst
from research_crew.tasks import task_1, task_2

sts = boto3.client("sts")

def upload_to_s3_and_get_url(file_path: str, bucket: str, object_key: str, expiry: int = 3600) -> str:
    s3 = boto3.client("s3")
    
    try:
        # Upload the file
        s3.upload_file(file_path, bucket, object_key)

        # Generate presigned URL
        url = s3.generate_presigned_url(
            ClientMethod='get_object',
            Params={'Bucket': bucket, 'Key': object_key},
            ExpiresIn=expiry
        )
        return url
    except ClientError as e:
        raise RuntimeError(f"S3 upload or presign failed: {e}")

# Run the crew
crew = Crew(
    agents=[researcher, analyst],
    tasks=[task_1, task_2],
    process=Process.sequential,
    verbose=True
)

if __name__ == "__main__":
    """
    Run the research crew.
    """
    inputs = {
        'topic': 'Artificial Intelligence'
    }
    os.makedirs("output", exist_ok=True)
    output_file = f"output/report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.md"

    result = crew.kickoff(inputs=inputs)

    # Save raw result to file
    with open(output_file, "w") as f:
        f.write(result.raw)
    
    # Upload and get URL
    bucket_name = f"crewaiagentcore-{sts.get_caller_identity()["Account"]}"
    object_key = os.path.basename(output_file)
    url = upload_to_s3_and_get_url(output_file, bucket_name, object_key)

    print("\n\n=== PRESIGNED URL ===\n")
    print(url)

### Invoking crew locally

Finally, we can use the CrewAI CLI to lockally kick off the crew. Alternatively, we could also simply run our local entrypoint main.py. This might take a few minutes. 

In [ ]:
!python local_crew.py

## Deploying multi-agent crew to Amazon Bedrock AgentCore

For production-grade agentic applications we will need to run our crew in the cloud. Therefor we will deploy our crew to Amazon Bedrock AgentCore. 

The architecture here will look as following:

<div style="text-align:left">
     <img src="images/architecture_local.png" width="60%"/>
</div>

Deploying the crew to AgentCore takes the following steps: 

### Remote entrypoint

First, we create a remote entrypoint. With AgentCore Runtime, we will decorate the invocation part of our agent with the @app.entrypoint decorator and have it as the entry point for our runtime. This also involves: 
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards                                                                                                                                                                        

In [ ]:
%%writefile crew.py
import datetime
import boto3
from botocore.exceptions import ClientError
import os

from crewai import Crew, Process
from research_crew.agents import researcher, analyst
from research_crew.tasks import task_1, task_2

sts = boto3.client("sts")

# ---------- Agentcore imports --------------------
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

#--------------Utility Functions---------------------

def upload_to_s3_and_get_url(file_path: str, bucket: str, object_key: str, expiry: int = 3600) -> str:
    s3 = boto3.client("s3")
    
    try:
        # Upload the file
        s3.upload_file(file_path, bucket, object_key)

        # Generate presigned URL
        url = s3.generate_presigned_url(
            ClientMethod='get_object',
            Params={'Bucket': bucket, 'Key': object_key},
            ExpiresIn=expiry
        )
        return url
    except ClientError as e:
        raise RuntimeError(f"S3 upload or presign failed: {e}")

#--------------App Entrypoint---------------------

@app.entrypoint
def agent_invocation(payload, context):
    """Handler for agent invocation"""
    print(f'Payload: {payload}')
    try: 
        # Extract user message from payload with default
        user_message = payload.get("prompt", "Artificial Intelligence in Healthcare")
        print(f"Processing topic: {user_message}")
        
        os.makedirs("output", exist_ok=True)
        output_file = f"output/report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.md"

        # Create crew instance and run synchronously
        crew = Crew(
            agents=[researcher, analyst],
            tasks=[task_1, task_2],
            process=Process.sequential,
            verbose=False
        )

        # Use synchronous kickoff instead of async - this avoids all event loop issues
        result = crew.kickoff(inputs={'topic': user_message})

        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        # Save raw result to file
        with open(output_file, "w") as f:
            f.write(result.raw)
        
        # Upload and get URL
        bucket_name = f"crewaiagentcore-{sts.get_caller_identity()["Account"]}"

        object_key = os.path.basename(output_file)
        url = upload_to_s3_and_get_url(output_file, bucket_name, object_key)
                
        return url
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return f"An error occurred: {str(e)}"

if __name__ == "__main__":
    app.run()

### Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

#### Creation of execution role for remote agentic workload

Then, we create a IAM execution role equipping our remote agentic workload with the required permissions to run. 

In [ ]:
print(f"Creating IAM role for {agent_name}...")
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

role_name = f"agentcore-{agent_name}-role"
policy_name = f"{role_name}-s3-access"

# Construct policy document
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
                "s3:GetObject"
            ],
            "Resource": [
                f"arn:aws:s3:::{bucket_name}/*"
            ]
        }
    ]
}

try:
    # Attach as an inline policy
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName=policy_name,
        PolicyDocument=json.dumps(policy_document)
    )
    print(f"✅ Attached inline S3 policy to role: {role_name}")
except iam_client.exceptions.NoSuchEntityException:
    print(f"❌ IAM role not found: {role_name}")
except Exception as e:
    raise RuntimeError(f"❌ Failed to attach policy: {e}")

print("IAM role created ✓")
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")
    

#### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

AgentCore configure is required to generate a Dockerfile holding a blueprint for the Docker container the workload will be running in and a .bedrock_agentcore.yaml holding the agentic workload's configuration. During the configure step, your docker file will be generated based on your application code.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
print(f"Using AWS region: {region}")

required_files = [
    "crew.py",
    "requirements.txt",
    "research_crew/agents.py",
    "research_crew/tasks.py",
]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

print("Configuring AgentCore Runtime...")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="crew.py",
    execution_role=agentcore_iam_role["Role"]["Arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
)
response

#### Launching agent to AgentCore Runtime: deploying the remote agentic workload

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime. AgentCore launch will then deploy the agentic workload to the cloud. This includes creating a Docker image and pushing it to ECR, as well as getting an endpoint ready for usage.


<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

When using the `agentcore_runtime.launch()` method, you can optionally pass `use_codebuild=True` to use [AWS CodeBuild](https://aws.amazon.com/codebuild/) instead of local Docker to build and push the image to Amazon ECR.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch(use_codebuild=True)

#### If you want to build the image using local Docker, run this command instead ####
# launch_result = agentcore_runtime.launch()print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

#### Checking for the AgentCore Runtime Status

Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
print(f"Initial status: {status}")

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")

print(f"Final status: {status}")

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it. Since this is a long running agent we are overwriting the default `retries`, `connect_timout`and `read_timeout`.

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
agent_arn = launch_result.agent_arn

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "Generative Ai"}),
)

# Process and print the response
if "text/event-stream" in boto3_response.get("contentType", ""):
  
    # Handle streaming response
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=10):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    print("\nComplete response:", "\n".join(content))

elif boto3_response.get("contentType") == "application/json":
    # Handle standard JSON response
    content = []
    for chunk in boto3_response.get("response", []):
        content.append(chunk.decode('utf-8'))
    print(json.loads(''.join(content)))
  
else:
    # Print raw response for other content types
    print(boto3_response)

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
print(f"ECR Repository URI: {launch_result.ecr_uri}")
print(f"Agent ID: {launch_result.agent_id}")
print(f"ECR Repository Name: {launch_result.ecr_uri.split('/')[1]}")
print(f"CodeBuild Project Name: {launch_result.codebuild_id.split(':')[0]}")

In [ ]:
# print("🗑️  Starting cleanup process...")

# print("Deleting AgentCore Runtime...")
# try:
#     runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#         agentRuntimeId=launch_result.agent_id,
#     )
#     print(f"Deleted agent runtime: {launch_result.agent_id}")
# except Exception as e:
#     print(f"Agent runtime {launch_result.agent_id} not found or already deleted: {e}")
#     print("You may need to manually clean up some resources.")

# print("Deleting ECR repository...")
# try:
#     response = ecr_client.delete_repository(
#         repositoryName=launch_result.ecr_uri.split("/")[1], force=True
#     )
#     print(f"Deleted ECR repository: {launch_result.ecr_uri.split('/')[1]}")
# except Exception as e:
#     print(
#         f"ECR repository {launch_result.ecr_uri.split('/')[1]} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# print("Deleting IAM role policies...")
# try:
#     policies = iam_client.list_role_policies(
#         RoleName=agentcore_iam_role["Role"]["RoleName"], MaxItems=100
#     )

#     for policy_name in policies["PolicyNames"]:
#         iam_client.delete_role_policy(
#             RoleName=agentcore_iam_role["Role"]["RoleName"], PolicyName=policy_name
#         )

#     iam_response = iam_client.delete_role(
#         RoleName=agentcore_iam_role["Role"]["RoleName"]
#     )
#     print(f"Deleted IAM role: {agentcore_iam_role['Role']['RoleName']}")
# except Exception as e:
#     print(
#         f"IAM role {agentcore_iam_role['Role']['RoleName']} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# try:
#     response = codebuild_client.delete_project(
#         name=launch_result.codebuild_id.split(":")[0]
#     )
#     print(f"Deleted CodeBuild project: {launch_result.codebuild_id.split(':')[0]}")
# except Exception as e:
#     print(
#         f"CodeBuild project {launch_result.codebuild_id.split(':')[0]} not found or already deleted: {e}"
#     )
#     print("You may need to manually clean up some resources.")

# s3 = boto3.resource("s3")
# bucket = s3.Bucket(bucket_name)

# try:
#     print(f"Deleting all contents in: {bucket_name}")
#     bucket.objects.all().delete()
    
#     print(f"Deleting bucket: {bucket_name}")
#     bucket.delete()
# except ClientError as e:
#     raise RuntimeError(f"Failed to delete bucket or contents: {e}")

# print("\n✅ Cleanup completed successfully!")

## Congratulations!